In [1]:
import os
import sys

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python executable:", sys.executable)
print("SPARK_LOCAL_IP:", os.environ["SPARK_LOCAL_IP"])

Python executable: C:\Users\zhoul\anaconda3\python.exe
SPARK_LOCAL_IP: 127.0.0.1


In [2]:
import pyspark

print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("PySpark version:", pyspark.__version__)

JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot\
PySpark version: 4.2.0


In [3]:
!java -version

openjdk version "17.0.20.1" 2026-08-18
OpenJDK Runtime Environment Temurin-17.0.20.1+1 (build 17.0.20.1+1)
OpenJDK 64-Bit Server VM Temurin-17.0.20.1+1 (build 17.0.20.1+1, mixed mode, sharing)


In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ITO5202 Assessment 1")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("Master:", sc.master)
print("Default parallelism:", sc.defaultParallelism)
print("Spark UI:", sc.uiWebUrl)

C:\Users\zhoul\anaconda3\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0
Master: local[*]
Default parallelism: 8
Spark UI: http://127.0.0.1:4040


In [5]:
spark.range(10).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



In [6]:
test_df = spark.createDataFrame(
    [
        (1, "Manhattan"),
        (2, "Queens"),
        (3, "Brooklyn"),
        (4, "Manhattan")
    ],
    ["id", "borough"]
)

test_df.groupBy("borough").count().show()

+---------+-----+
|  borough|count|
+---------+-----+
|Manhattan|    2|
|   Queens|    1|
| Brooklyn|    1|
+---------+-----+



In [7]:
import sys
print(sys.version)

3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]


In [8]:
import sys

print("Jupyter Python:", sys.executable)
print("Spark Python:", sc.pythonExec)

Jupyter Python: C:\Users\zhoul\anaconda3\python.exe
Spark Python: C:\Users\zhoul\anaconda3\python.exe


In [9]:
sc.parallelize([1, 2, 3, 4]).collect()

[1, 2, 3, 4]

# Part A: Analytical Query Design and Implementation

## 1. Dataset Loading and Initial Exploration

In [10]:
spark.conf.set("spark.sql.session.timeZone", "America/New_York")

In [11]:
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    DoubleType,
    StringType,
    TimestampType,
    IntegerType
)

trip_schema_jan = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True)
])

In [12]:
zone_schema = StructType([
    StructField("LocationID", IntegerType(), True),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True)
])

In [13]:
zone_df = (
    spark.read
    .option("header", True)
    .schema(zone_schema)
    .csv("data/taxi_zone_lookup.csv")
)

zone_df.show(10)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 10 rows


In [14]:
zone_df.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [15]:
jan_check_df = spark.read.parquet(
    "data/yellow_tripdata_2023-01.parquet"
)

jan_check_df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [16]:
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    DoubleType,
    StringType,
    TimestampNTZType
)

jan_source_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True)
])

In [17]:
jan_df = (
    spark.read
    .schema(jan_source_schema)
    .parquet("data/yellow_tripdata_2023-01.parquet")
)

jan_df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [18]:
from pyspark.sql.functions import col

jan_standardised_df = jan_df.select(
    col("VendorID").cast("long").alias("VendorID"),
    col("tpep_pickup_datetime").cast("timestamp").alias("tpep_pickup_datetime"),
    col("tpep_dropoff_datetime").cast("timestamp").alias("tpep_dropoff_datetime"),
    col("passenger_count").cast("long").alias("passenger_count"),
    col("trip_distance").cast("double").alias("trip_distance"),
    col("RatecodeID").cast("long").alias("RatecodeID"),
    col("store_and_fwd_flag").cast("string").alias("store_and_fwd_flag"),
    col("PULocationID").cast("long").alias("PULocationID"),
    col("DOLocationID").cast("long").alias("DOLocationID"),
    col("payment_type").cast("long").alias("payment_type"),
    col("fare_amount").cast("double").alias("fare_amount"),
    col("extra").cast("double").alias("extra"),
    col("mta_tax").cast("double").alias("mta_tax"),
    col("tip_amount").cast("double").alias("tip_amount"),
    col("tolls_amount").cast("double").alias("tolls_amount"),
    col("improvement_surcharge").cast("double").alias("improvement_surcharge"),
    col("total_amount").cast("double").alias("total_amount"),
    col("congestion_surcharge").cast("double").alias("congestion_surcharge"),
    col("airport_fee").cast("double").alias("airport_fee")
)

In [19]:
jan_standardised_df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [20]:
jan_standardised_df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2023-01-01 00:32:10 |2023-01-01 00:40:36  |1              |0.97         |1         |N                 |161         |141         |2           |9.3        |1.0  |0.5    |0.0      

In [21]:
feb_check_df = spark.read.parquet(
    "data/yellow_tripdata_2023-02.parquet"
)

feb_check_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [22]:
for month in range(1, 13):
    path = f"data/yellow_tripdata_2023-{month:02d}.parquet"

    print(f"\nMonth: {month:02d}")
    spark.read.parquet(path).printSchema()


Month: 01
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)


Month: 02
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-

In [23]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    LongType,
    DoubleType,
    StringType,
    TimestampNTZType
)

later_source_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True)
])

In [24]:
from pyspark.sql.functions import col

def standardise_trip_df(df, airport_column):
    return df.select(
        col("VendorID").cast("long").alias("VendorID"),
        col("tpep_pickup_datetime").cast("timestamp").alias("tpep_pickup_datetime"),
        col("tpep_dropoff_datetime").cast("timestamp").alias("tpep_dropoff_datetime"),
        col("passenger_count").cast("long").alias("passenger_count"),
        col("trip_distance").cast("double").alias("trip_distance"),
        col("RatecodeID").cast("long").alias("RatecodeID"),
        col("store_and_fwd_flag").cast("string").alias("store_and_fwd_flag"),
        col("PULocationID").cast("long").alias("PULocationID"),
        col("DOLocationID").cast("long").alias("DOLocationID"),
        col("payment_type").cast("long").alias("payment_type"),
        col("fare_amount").cast("double").alias("fare_amount"),
        col("extra").cast("double").alias("extra"),
        col("mta_tax").cast("double").alias("mta_tax"),
        col("tip_amount").cast("double").alias("tip_amount"),
        col("tolls_amount").cast("double").alias("tolls_amount"),
        col("improvement_surcharge").cast("double").alias("improvement_surcharge"),
        col("total_amount").cast("double").alias("total_amount"),
        col("congestion_surcharge").cast("double").alias("congestion_surcharge"),
        col(airport_column).cast("double").alias("airport_fee")
    )

In [25]:
jan_raw_df = (
    spark.read
    .schema(jan_source_schema)
    .parquet("data/yellow_tripdata_2023-01.parquet")
)

jan_df = standardise_trip_df(
    jan_raw_df,
    "airport_fee"
)

In [26]:
monthly_dfs = [jan_df]

for month in range(2, 13):
    path = f"data/yellow_tripdata_2023-{month:02d}.parquet"

    month_raw_df = (
        spark.read
        .schema(later_source_schema)
        .parquet(path)
    )

    month_df = standardise_trip_df(
        month_raw_df,
        "Airport_fee"
    )

    monthly_dfs.append(month_df)

In [27]:
from functools import reduce

trips_df = reduce(
    lambda df1, df2: df1.unionByName(df2),
    monthly_dfs
)

In [28]:
trips_df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [29]:
trips_df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2023-01-01 00:32:10 |2023-01-01 00:40:36  |1              |0.97         |1         |N                 |161         |141         |2           |9.3        |1.0  |0.5    |0.0      

In [30]:
print("Number of monthly DataFrames:", len(monthly_dfs))
print("Total trip records:", trips_df.count())

Number of monthly DataFrames: 12
Total trip records: 38310226


In [31]:
spark.conf.set("spark.sql.session.timeZone", "America/New_York")

In [32]:
from pyspark.sql.functions import (
    col,
    to_date,
    hour,
    dayofweek,
    unix_timestamp
)

trips_derived_df = (
    trips_df
    .withColumn(
        "pickup_date",
        to_date(col("tpep_pickup_datetime"))
    )
    .withColumn(
        "pickup_hour",
        hour(col("tpep_pickup_datetime"))
    )
    .withColumn(
        "day_of_week",
        dayofweek(col("tpep_pickup_datetime"))
    )
    .withColumn(
        "trip_duration_minutes",
        (
            unix_timestamp(col("tpep_dropoff_datetime"))
            - unix_timestamp(col("tpep_pickup_datetime"))
        ) / 60.0
    )
    .withColumn(
        "pickup_epoch_seconds",
        unix_timestamp(col("tpep_pickup_datetime")).cast("long")
    )
)

In [33]:
trips_derived_df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "pickup_date",
    "pickup_hour",
    "day_of_week",
    "trip_duration_minutes",
    "pickup_epoch_seconds"
).show(10, truncate=False)

+--------------------+---------------------+-----------+-----------+-----------+---------------------+--------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|pickup_date|pickup_hour|day_of_week|trip_duration_minutes|pickup_epoch_seconds|
+--------------------+---------------------+-----------+-----------+-----------+---------------------+--------------------+
|2023-01-01 00:32:10 |2023-01-01 00:40:36  |2023-01-01 |0          |1          |8.433333333333334    |1672551130          |
|2023-01-01 00:55:08 |2023-01-01 01:01:27  |2023-01-01 |0          |1          |6.316666666666666    |1672552508          |
|2023-01-01 00:25:04 |2023-01-01 00:37:49  |2023-01-01 |0          |1          |12.75                |1672550704          |
|2023-01-01 00:03:48 |2023-01-01 00:13:25  |2023-01-01 |0          |1          |9.616666666666667    |1672549428          |
|2023-01-01 00:10:29 |2023-01-01 00:21:19  |2023-01-01 |0          |1          |10.833333333333334   |1672549829          |
|2023-01

In [34]:
print("Taxi zone records:", zone_df.count())
zone_df.show(5, truncate=False)

Taxi zone records: 265
+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 5 rows


In [35]:
pickup_zone_df = zone_df.select(
    col("LocationID").alias("PULocationID"),
    col("Borough").alias("pickup_borough"),
    col("Zone").alias("pickup_zone"),
    col("service_zone").alias("pickup_service_zone")
)

dropoff_zone_df = zone_df.select(
    col("LocationID").alias("DOLocationID"),
    col("Borough").alias("dropoff_borough"),
    col("Zone").alias("dropoff_zone"),
    col("service_zone").alias("dropoff_service_zone")
)

In [36]:
trips_joined_df = (
    trips_derived_df
    .join(
        pickup_zone_df,
        on="PULocationID",
        how="left"
    )
    .join(
        dropoff_zone_df,
        on="DOLocationID",
        how="left"
    )
)

In [37]:
trips_joined_df.select(
    "PULocationID",
    "pickup_borough",
    "pickup_zone",
    "DOLocationID",
    "dropoff_borough",
    "dropoff_zone"
).show(10, truncate=False)

+------------+--------------+---------------------+------------+---------------+-----------------------------------+
|PULocationID|pickup_borough|pickup_zone          |DOLocationID|dropoff_borough|dropoff_zone                       |
+------------+--------------+---------------------+------------+---------------+-----------------------------------+
|161         |Manhattan     |Midtown Center       |141         |Manhattan      |Lenox Hill West                    |
|43          |Manhattan     |Central Park         |237         |Manhattan      |Upper East Side South              |
|48          |Manhattan     |Clinton East         |238         |Manhattan      |Upper West Side North              |
|138         |Queens        |LaGuardia Airport    |7           |Queens         |Astoria                            |
|107         |Manhattan     |Gramercy             |79          |Manhattan      |East Village                       |
|161         |Manhattan     |Midtown Center       |137         |

In [38]:
print("Records before joins:", trips_derived_df.count())
print("Records after joins:", trips_joined_df.count())

Records before joins: 38310226
Records after joins: 38310226


In [39]:
from pyspark.sql.functions import col, sum as spark_sum

trips_joined_df.select(
    spark_sum(
        col("pickup_borough").isNull().cast("int")
    ).alias("unmatched_pickup_locations"),

    spark_sum(
        col("dropoff_borough").isNull().cast("int")
    ).alias("unmatched_dropoff_locations")
).show()

+--------------------------+---------------------------+
|unmatched_pickup_locations|unmatched_dropoff_locations|
+--------------------------+---------------------------+
|                         0|                          0|
+--------------------------+---------------------------+



In [40]:
trips_joined_df.filter(
    col("pickup_borough").isNull()
).groupBy(
    "PULocationID"
).count().orderBy(
    col("count").desc()
).show(20)

+------------+-----+
|PULocationID|count|
+------------+-----+
+------------+-----+



In [41]:
trips_joined_df.filter(
    col("dropoff_borough").isNull()
).groupBy(
    "DOLocationID"
).count().orderBy(
    col("count").desc()
).show(20)

+------------+-----+
|DOLocationID|count|
+------------+-----+
+------------+-----+



In [42]:
from pyspark.sql.functions import col, sum as spark_sum

trips_joined_df.select(
    spark_sum((col("trip_distance") <= 0).cast("int")).alias("non_positive_distance"),
    spark_sum((col("fare_amount") < 0).cast("int")).alias("negative_fare"),
    spark_sum((col("trip_duration_minutes") <= 0).cast("int")).alias("non_positive_duration")
).show()

+---------------------+-------------+---------------------+
|non_positive_distance|negative_fare|non_positive_duration|
+---------------------+-------------+---------------------+
|               773457|       381650|                15569|
+---------------------+-------------+---------------------+



In [43]:
from pyspark.sql.functions import col

invalid_df = trips_joined_df.filter(
    (col("trip_distance") <= 0) |
    (col("fare_amount") < 0) |
    (col("trip_duration_minutes") <= 0)
)

print("Records failing at least one validity check:", invalid_df.count())

Records failing at least one validity check: 1117974


In [44]:
trips_joined_df.filter(
    (col("pickup_date") < "2023-01-01") |
    (col("pickup_date") > "2023-12-31")
).select(
    "pickup_date"
).groupBy(
    "pickup_date"
).count().orderBy(
    "pickup_date"
).show(50)

+-----------+-----+
|pickup_date|count|
+-----------+-----+
| 2001-01-01|    6|
| 2002-12-31|   11|
| 2003-01-01|    6|
| 2008-12-31|   23|
| 2009-01-01|   15|
| 2014-11-19|    1|
| 2022-10-24|    4|
| 2022-10-25|    7|
| 2022-12-31|   25|
| 2024-01-01|    2|
| 2024-01-03|    4|
+-----------+-----+



In [45]:
valid_trips_df = trips_joined_df.filter(
    (col("pickup_date") >= "2023-01-01") &
    (col("pickup_date") <= "2023-12-31") &
    (col("trip_distance") > 0) &
    (col("fare_amount") >= 0) &
    (col("trip_duration_minutes") > 0)
)

In [46]:
total_records = trips_joined_df.count()
valid_records = valid_trips_df.count()

print("Total records:", total_records)
print("Valid records:", valid_records)
print("Records excluded:", total_records - valid_records)

Total records: 38310226
Valid records: 37192160
Records excluded: 1118066
